In [8]:
# transferable code functions
import numpy as np

def padding(layer:np.ndarray, mode:str = 'zero'):


    padded_ly_size = (layer.shape[0]+2, layer.shape[1]+2)
    padded_ly = np.zeros(padded_ly_size)
    padded_ly[1:-1,1:-1] = layer
   
    if mode == 'continue':
        padded_ly[0,1:-1] = layer[0,:]
        padded_ly[-1,1:-1] = layer[-1,:]

        padded_ly[1:-1,0] = layer[:,0]
        padded_ly[1:-1,-1] = layer[:,-1]

        padded_ly[0,0] = layer[0,0]
        padded_ly[0,-1] = layer[0,-1]
        padded_ly[-1,0] = layer[-1,0]
        padded_ly[-1,-1] = layer[-1,-1]

    return(padded_ly)



In [9]:
import hydra
from omegaconf import DictConfig, OmegaConf
cfg = OmegaConf.load("model.yaml")


In [16]:
#todo I'd like to create a version of this that can have dynamically sized convolutional layers
import numpy as np
from dataclasses import dataclass
from omegaconf import DictConfig

def build_index_lookup(cfg: DictConfig):
    """
    Given a loaded OmegaConf config with a 'layers' section,
    build a lookup table: index -> (layer_name, layer_data).
    """
    blocks = cfg.blocks

    index_lookup = {
        block_data.index: (block_name, block_data)
        for block_name, block_data in blocks.items()
    }
    return index_lookup


print("start")


class Layer:
    def __init__(self, ltype:str, index:int,  activations: np.ndarray, z_values: np.ndarray, dims:int, shape: tuple):
        self.activations = activations
        self.z_values = z_values
        self.ltype = ltype
        self.shape = shape
        self.dims = dims
        self.index = index

    @classmethod
    def from_shape(cls, layer_shape, l_type, index, dtype=np.float32, **kwargs):
        activations = np.zeros(layer_shape, dtype=dtype)
        z_vals = np.zeros(layer_shape, dtype=dtype)
        shape = activations.shape
        ltype = l_type
        dims = len(shape)
        index = index
        return cls(ltype, index, activations, z_vals, dims, shape, **kwargs)


class Conv_Block:
    def __init__(self, num_filters, kernel_shape, stride, layer_shape, index):
        self.num_filters = num_filters
        self.kernel_shape = kernel_shape
        self.stride = stride
        self.layer_shape = layer_shape
        self.index = index

        self.w_type = "conv"
        self.ly_dim = len(layer_shape)
        self.filters = self.init_filters(self.num_filters, self.kernel_shape, init = True)
        self.feature_maps = self._init_feature_maps()
        self.layer, self.z_values = self._init_layer()

    def _init_feature_maps(self):
        feature_maps = np.zeros(shape=(self.num_filters, *self.layer_shape))
        return(feature_maps)

    def _init_layer(self):
        l_activations = np.zeros(self.layer_shape)
        z_values = np.zeros(self.layer_shape)
        return(l_activations, z_values)

    @staticmethod    
    def init_filters(num_filters:int, kernel_shape:tuple, init=True):
        if init:
            filters = np.random.uniform(-1,1, size=(*kernel_shape, num_filters))
        else:
            filters = np.zeros(*kernel_shape, num_filters)
        return (filters)

    
class FC_Weights:
    def __init__(self, weights: np.ndarray, index, w_type: str, ly_dim: int, shape: tuple):
        self.weights = weights
        self.shape = shape
        self.w_type = w_type
        self.ly_dim = ly_dim
        self.index = index

    @classmethod
    def full_connected_from_shape(cls, index, prev_ly_shape: tuple, curr_ly_shape: tuple, dtype=np.float32, init=True, **kwargs):
        if init:
            #ly_weights = np.random.rand(prev_ly_size, curr_ly_size)
            ly_weights = np.random.uniform(-1,1, size=(*prev_ly_shape, *curr_ly_shape))
        else:
            ly_weights = np.zeros(shape=(*prev_ly_shape, *curr_ly_shape))
        w_type = "fc"
        ly_dim = len(curr_ly_shape)
        shape = ly_weights.shape
        index = index
        return cls(ly_weights, index, w_type, ly_dim, shape)


class Biases:
    def __init__(self, biases: np.ndarray, index, dims:int, shape:tuple):
        self.biases = biases
        self.dims = dims
        self.shape = shape
        self.index = index

    @classmethod
    def from_shape(cls, index, curr_ly_shape:tuple, dtype=np.float32, init=True, **kwargs):
        if init:
            biases = np.random.rand(*curr_ly_shape)
        else:
            biases = np.zeros(curr_ly_shape)
        
        dims = len(curr_ly_shape)
        shape = curr_ly_shape
        index = index
        return cls(biases, index, dims, shape)

    def from_dynamic_shape(cls, kernel_shape, prev_ly_shape, curr_ly_shape, dtype=np.float32, init=True, **kwargs):
        print("not yet implemented :(")


class NN:
    def __init__(self, config:DictConfig, layers: list, weights: list, biases: list):
        self.config = config
        self.layers = layers # layers hold both actual activations and z values
        self.weights = weights
        self.biases = biases

    @classmethod
    def create_network(cls, cfg:DictConfig, **kwargs):
        index_to_block = build_index_lookup(cfg)

        blocks = []
        for items in cfg.blocks:
            print(items)
            block_name, block_data = index_to_block[items.index]

            if items.type == "input":
                blocks.append(" ")

            if items.type == "fc":
                blocks.append(" ")
            if items.type == "conv":
                fltr = items.filters
                blocks.append(Conv_Block(fltr.filter_num, fltr.kernel_shape, fltr.stride, items.shape, items.index))

        layers = []
        for items in cfg.layers:
            layers.append(Layer.from_shape(tuple(cfg.layers[items].shape), cfg.layers[items].type, cfg.layers[items].index))

        biases =[]
        weights = []
        for items in layers: 
            if items.index != 0:
                biases.append(Biases.from_shape(items.index, items.shape))

            layer_name, layer_data = index_to_layer[items.index]
            if items.index != 0:
                if items.ltype == 'fc':
                    weights.append(Weights.full_connected_from_shape(items.index, prev_ly.shape, items.shape))
                if items.ltype == "conv2D":
                    weights.append(Weights.filters_from_shape(items.index, layer_data.weights.kernel_size, layer_data.weights.filter_num))
            prev_ly = items


        return cls(cfg, layers, weights, biases)

    



def forward(input_vals:np.ndarray, net:NN):
    net.layers[0] = input_vals


    for index in range(1, len(net.layers), 1):
        shape = net.layers[index].activations.shape
        new_z_ly = np.zeros(shape)
        new_ly   = np.zeros(shape)
        #net.z_layers[index] = np.dot(net.layers[(index-1)], net.weights[index-1]) + net.biases[index-1]
        #todo: find a preforment way to do this!
        for i in shape[0]:
            for j in shape[1]:
                new_z_ly[i,j] = net.layers[(index - 1)].activations[i,j]* net.weights[index-1].weights[i,j]
                



#todo:
    #figure out network autocreation
    #figure out padding algorrithm
    #figure out down sizing

start


In [17]:
cfg = OmegaConf.load("model.yaml")

model = NN.create_network(cfg)

input_ly


KeyError: <built-in method index of str object at 0x7df3d274d1b0>

In [57]:
item = model.layers
print("layers")
for items in item:
    print(items.shape, type(items.activations), items.index, items.ltype)
item = model.biases
print("biases")
for items in item:
    print(items.shape, type(items.biases), items.index)
item = model.weights
print("weights")
for items in item:
    print(items.shape, type(items.weights), items.index, items.w_type)




layers
(28, 28) <class 'numpy.ndarray'> 0 input
(28, 28) <class 'numpy.ndarray'> 1 conv2D
(14, 14) <class 'numpy.ndarray'> 2 conv2D
(14, 14) <class 'numpy.ndarray'> 3 conv2D
(10,) <class 'numpy.ndarray'> 4 fc
biases
(28, 28) <class 'numpy.ndarray'> 1
(14, 14) <class 'numpy.ndarray'> 2
(14, 14) <class 'numpy.ndarray'> 3
(10,) <class 'numpy.ndarray'> 4
weights
(3, 3, 27) <class 'numpy.ndarray'> 1 conv
(3, 3, 27) <class 'numpy.ndarray'> 2 conv
(3, 3, 27) <class 'numpy.ndarray'> 3 conv
(14, 14, 10) <class 'numpy.ndarray'> 4 fc


In [ ]:
cfg = OmegaConf.load("model.yaml")
print(type(cfg))
print(cfg)
examine_1 = cfg.model.layers.input_ly.shape
print("\nexamine_1: ")
print(examine_1)
print(type(examine_1))
examine_2 = tuple(cfg.model.layers.input_ly.shape)
print("\nexamine_2: ")
print(examine_2)
print(type(examine_2))
#next to figure out how to handle .yaml files and DictConfig files

#net = NN.create_network(cfg)


In [ ]:
import numpy as np
filters  = {"prev_ly_size":28,
            "kernel_size": 3,
            "stride" : 1}
l1 = np.zeros(shape=(filters["prev_ly_size"],))
l11 = np.zeros(shape=(filters["prev_ly_size"],))

l2 = []
l1[0] = 1
for index, values in enumerate(l1):
    if index%filters["stride"] == 0:
        l1[index] = 1


    if l1[index] == 1:
        for x in range(filters["kernel_size"]): 
            y=x+1
            try:
                l11[index + (y - filters["kernel_size"]//2)] = l11[index + (y - filters["kernel_size"]//2)] + 1
            except IndexError as error:
                print("had an index error, continuing")
print(l1,"\n",l11)